# Retail Shelf Object Detection - Reference Solution

Prior-based detection baseline using category statistics from training annotations.

In [ ]:
import csv, random
from collections import defaultdict, Counter

random.seed(0)

CATS = ['cereal_box','beverage_can','water_bottle','snack_bag','dairy_carton',
        'cleaning_product','frozen_food','condiment_jar','bread_loaf','canned_goods']

# Load prior box sizes
sizes = defaultdict(list)
with open('dataset/public/train_annotations.csv','r') as f:
    for row in csv.DictReader(f):
        sizes[row['category']].append((float(row['bbox_w']), float(row['bbox_h'])))

cat_priors = {cat: (sum(v[0] for v in vals)/len(vals), sum(v[1] for v in vals)/len(vals)) 
              for cat, vals in sizes.items()}
print('Loaded priors for', len(cat_priors), 'categories')

In [ ]:
# Build section -> category distribution
img_section = {}
with open('dataset/public/images.csv','r') as f:
    for row in csv.DictReader(f): img_section[row['image_id']] = row['shelf_section']

section_dist = defaultdict(Counter)
with open('dataset/public/train_annotations.csv','r') as f:
    for row in csv.DictReader(f):
        section_dist[img_section.get(row['image_id'],'grocery')][row['category']] += 1

with open('dataset/public/test_images.csv','r') as f:
    test_imgs = list(csv.DictReader(f))

results = []
for img in test_imgs:
    dist = section_dist.get(img['shelf_section'], {})
    cats, wts = zip(*dist.items()) if dist else (CATS, [1]*len(CATS))
    for _ in range(random.randint(3,8)):
        cat = random.choices(list(cats), weights=list(wts))[0]
        pw, ph = cat_priors.get(cat, (0.10, 0.15))
        bw = pw * random.uniform(0.8, 1.2)
        bh = ph * random.uniform(0.8, 1.2)
        results.append({'image_id': img['image_id'], 'category': cat,
                        'bbox_x': round(random.uniform(0, 1-bw), 4),
                        'bbox_y': round(random.uniform(0, 1-bh), 4),
                        'bbox_w': round(bw, 4), 'bbox_h': round(bh, 4),
                        'score': round(random.uniform(0.4, 0.9), 3)})

with open('predictions.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=['image_id','category','bbox_x','bbox_y','bbox_w','bbox_h','score'])
    w.writeheader(); w.writerows(results)
print(f'Saved {len(results)} predictions')